[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/MapAnything_Colab.ipynb)  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/MapAnything_Colab.ipynb)


# 🗺️ MapAnything — Universal 3DGS-from-Images (Meta, Apache 2.0)

**[MapAnything](https://map-anything.github.io/)** is Meta's universal 3D reconstruction
framework (3DV 2026). One feed-forward transformer handles **12+ 3D reconstruction
tasks**: image-only SfM, image+pose, image+intrinsics+depth, registration, depth
completion, monocular metric depth, and more. Output is a **metric 3D point cloud +
camera poses** which we feed into `gsplat` to produce a real 3DGS scene.

* **Paper:** [arXiv 2509.13414](https://arxiv.org/abs/2509.13414) &nbsp;·&nbsp;
  **Project:** [map-anything.github.io](https://map-anything.github.io/) &nbsp;·&nbsp;
  **Code:** [github.com/facebookresearch/map-anything](https://github.com/facebookresearch/map-anything) &nbsp;·&nbsp;
  **Weights (Apache 2.0):** [🤗&nbsp;facebook/map-anything-apache](https://huggingface.co/facebook/map-anything-apache)
* **Authors:** Keetha, Müller, Schönberger, Porzi, Zhang, Fischer, Knapitsch, Zauss, Weber, Antunes, Luiten, Lopez-Antequera, Rota Bulò, Richardt, Ramanan, Scherer, Kontschieder (Meta + CMU)
* **Conference:** 3DV 2026
* **License:** **Apache 2.0** for both the code AND the weights (the `facebook/map-anything-apache` repo). A CC-BY-NC variant exists at `facebook/map-anything` for researchers — we use the Apache variant for full commercial-OK use.

## How it differs from our other 3DGS notebooks

| Notebook | Inputs | Approach | License |
|---|---|---|---|
| **TripoSplat** | 1 image | Generative prior, feed-forward | MIT |
| **NoPoSplat** | 2-3 unposed photos | Pose-free, feed-forward | MIT (+ MASt3R backbone CC BY-NC-SA) |
| **Wild GS** | Video / image folder | MASt3R pose + 3DGS optimization | CC BY-NC-SA + INRIA non-commercial |
| **MapAnything (this)** | Any 1+ images; optional poses/intrinsics/depth | Universal feed-forward transformer, then gsplat | **Apache 2.0** (commercial-OK) |

The standout property of MapAnything is **universality**: you can give it 2 images, 20
images, an image + known depth from a depth sensor, an image + known camera poses from
COLMAP, or a video without any calibration at all — and it always produces a metric 3D
reconstruction in a single feed-forward pass.

## Pipeline

```
images (and optional poses/depth/intrinsics)
       ↓
MapAnything (universal transformer, 1 forward pass)
       ↓
metric point cloud + camera poses + intrinsics + depth
       ↓
write COLMAP-format (cameras.txt, images.txt, points3D.ply)
       ↓
gsplat (1-3 min training on T4)
       ↓
3DGS .ply (SOG/SPLAT/SPZ after SplatTransform_Colab STEP 3)
```

## Requirements
* **GPU:** NVIDIA, ≥ 6 GB VRAM (T4 15 GB works with `minibatch_size=1`)
* **RAM:** ≥ 12 GB
* **Disk:** ≈ 8 GB free (PyTorch + CUDA + 4.47 GB MapAnything + 1.1 GB DINOv2-giant + working space)
* **Time on first run:** 8-12 min (PyTorch + uniception + MapAnything + DINOv2-giant)
* **Time on subsequent runs:** 2-3 min (everything cached in your Drive)
* **Per-scene runtime:** ~20-60 seconds for 10 images at 518² on T4 (one forward pass)
* **Followed by gsplat:** +1-3 min depending on iteration count

## Where it fits in our 3DGS pipeline
```
MapAnything (this notebook, Apache 2.0)
   →  .ply + COLMAP cameras
   →  SplatTransform_Colab STEP 3  →  SOG/SPLAT/SPZ/GLB
   →  Asset_Library_Browser_Colab
   →  Three.js / WebGPU game engine
```


In [ ]:
#@title STEP 1 — Install dependencies + clone repo
"""
• Detects GPU + CUDA, installs torch 2.4.1+cu121, torchvision 0.19.1
• git clone --depth=1 facebookresearch/map-anything (Apache 2.0)
• pip install uniception==0.1.7 + the lightweight inference deps
  (no colmap / gradio extras — we use our own gsplat / Colab forms)
• MapAnything model + DINOv2-giant backbone cache to Drive
• PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True for memory efficiency
  (recommended by Meta for large view counts; reduces fragmentation)
"""
import os, sys, time, json, subprocess, shutil, pathlib, traceback
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

print('='*72)
print('MapAnything — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda or "none"})')
    print(f'  CUDA avail   : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f'  GPU {i}        : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected — MapAnything is unusable on CPU')
except ImportError:
    torch = None
    print('  torch not yet installed (will be installed below)')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/MapAnything')
    drive_root.mkdir(parents=True, exist_ok=True)
    print(f'  Drive cache  : {drive_root}')
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    os.environ['TORCH_HOME'] = str(drive_root / 'torch')
    os.environ['XDG_CACHE_HOME'] = str(drive_root / 'xdg')
else:
    drive_root = pathlib.Path('/content/_mapanything_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['TORCH_HOME'] = str(drive_root / 'torch')
    print(f'  Local cache  : {drive_root}  (lost on runtime disconnect)')

REPO_DIR = drive_root / 'map-anything'
t_total = time.time()

# 1. PyTorch ──────────────────────────────────────────────────
if torch is None or not torch.cuda.is_available() or not torch.__version__.startswith('2.4'):
    print('\n[1/5] Installing PyTorch 2.4.1 + cu121 ...')
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
         'torch==2.4.1', 'torchvision==0.19.1', 'torchaudio==2.4.1',
         '--index-url', 'https://download.pytorch.org/whl/cu121'],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  ERROR:', r.stderr[-1500:])
        raise RuntimeError('PyTorch install failed')
    print(f'  PyTorch installed in {time.time()-t0:.0f}s')

# 2. Clone the MapAnything repo ─────────────────────────────────
if not (REPO_DIR / '.git').exists():
    print(f'\n[2/5] Cloning facebookresearch/map-anything into {REPO_DIR} ...')
    t0 = time.time()
    r = subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/facebookresearch/map-anything.git',
         str(REPO_DIR)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  git clone failed:', r.stderr[-500:])
        raise RuntimeError('git clone failed')
    print(f'  cloned in {time.time()-t0:.0f}s')
else:
    print(f'\n[2/5] Reusing cached repo at {REPO_DIR}')

sys.path.insert(0, str(REPO_DIR))

# 3. Install uniception (the only non-trivial dep) + lightweight extras
print('\n[3/5] Installing uniception + inference deps ...')
t0 = time.time()
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     'uniception==0.1.7', 'safetensors', 'trimesh', 'huggingface_hub',
     'opencv-python-headless==4.10.0.84', 'pillow-heif', 'plyfile',
     'python-box', 'hydra-core', 'orjson', 'natsort',
     'rerun-sdk==0.24.1', 'tensorboard', 'einops'],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
print(f'  inference deps installed in {time.time()-t0:.0f}s')

# 4. Install gsplat for the 3DGS training step (uses the COLMAP output)
print('\n[4/5] Installing gsplat for 3DGS training ...')
t0 = time.time()
try:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         'gsplat==1.3.0', 'tyro', 'open3d'],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    print(f'  gsplat installed in {time.time()-t0:.0f}s')
except Exception as e:
    print(f'  gsplat install failed: {e}\n  (will retry in STEP 4)')

# 5. Download MapAnything model + DINOv2-giant (torch.hub) ───────
print('\n[5/5] Downloading MapAnything + DINOv2-giant to Drive ...')
from tqdm.auto import tqdm
t0 = time.time()
from huggingface_hub import snapshot_download
model_dir = drive_root / 'map-anything-apache'
if (model_dir / 'model.safetensors').exists() and \
   (model_dir / 'model.safetensors').stat().st_size > 4_000_000_000:
    print(f'  cached : map-anything-apache  ({(model_dir / "model.safetensors").stat().st_size//(1024*1024)} MB)')
else:
    print('  download : map-anything-apache/model.safetensors  (≈4.5 GB, 7-10 min)')
    snapshot_download(
        repo_id='facebook/map-anything-apache',
        local_dir=str(model_dir),
        allow_patterns=['model.safetensors', 'config.json', 'README.md'],
    )
    print(f'    -> {model_dir}')

# DINOv2-giant (torch.hub) — one-time download. The model fetches it from
# facebookresearch/dinov2 via torch.hub during MapAnything.from_pretrained(...).
torch_hub_dir = pathlib.Path(os.environ['TORCH_HOME']) / 'hub'
torch_hub_dir.mkdir(parents=True, exist_ok=True)
if any(torch_hub_dir.rglob('*giant*')):
    print('  cached : DINOv2-giant (torch.hub)')
else:
    print('  DINOv2-giant will be downloaded on first from_pretrained() (≈1.1 GB, 1-2 min)')
print(f'  total setup time so far: {time.time()-t0:.0f}s')

print()
print('='*72)
print(f'  STEP 1 complete  (total {time.time()-t_total:.0f}s)')
print('  Next: run STEP 2 (imports + lazy model load)')
print('='*72)


In [ ]:
#@title STEP 2 — Imports & lazy MapAnything model load
"""
Imports the MapAnything package (one-time cost). Builds the model via
`MapAnything.from_pretrained(..., cache_dir=drive_root)` which auto-downloads
DINOv2-giant on first call. The model is cached so subsequent Gradio clicks are instant.
Defines:
  • `infer(images, **kwargs)` → list of per-view prediction dicts
  • `write_colmap(predictions, output_dir, ...)` → COLMAP-format dataset
  • `save_glb(predictions, output_dir)` → dense point cloud as .glb
  • `run_full_pipeline(images, output_dir, **kwargs)` → both + gsplat
All upstream `model.infer()` params are exposed:
  apply_mask, mask_edges, apply_confidence_mask, confidence_percentile,
  use_multiview_confidence, use_amp, amp_dtype, minibatch_size,
  memory_efficient_inference, ignore_calibration_inputs, ignore_depth_inputs,
  ignore_pose_inputs, ignore_depth_scale_inputs, ignore_pose_scale_inputs.
"""
import sys, os, time, json, pathlib, warnings, shutil
warnings.filterwarnings('ignore')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

REPO_DIR = pathlib.Path(os.environ.get('AEI_MA_REPO',
                    str(pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/MapAnything/map-anything'))))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f'  repo        : {REPO_DIR}')
print(f'  exists      : {REPO_DIR.exists()}')
print()

import torch
import numpy as np
print('  Loading mapanything package (one-time cost) ...')
t0 = time.time()
try:
    from mapanything.models import MapAnything
    print(f'  mapanything.models.MapAnything ✓')
except Exception as e:
    print(f'  mapanything.models.MapAnything ✗  {type(e).__name__}: {e}')
    raise
from mapanything.utils.image import load_images
print(f'  mapanything.utils.image.load_images ✓')
print(f'  total import: {time.time()-t0:.1f}s')
print()

MODEL_REPO_ID = 'facebook/map-anything-apache'  # Apache 2.0 (commercial OK)
MODEL_DIR = drive_root / 'map-anything-apache'
MAP_RES = 518  # MapAnything's fixed input resolution (DINOv2 patch_size=14)
MAP_PATCH = 14

# ── Lazy model cache ────────────────────────────────────
_MODEL_CACHE = {}
def get_model(force_reload=False, amp_dtype='bf16', minibatch_size=1):
    key = (amp_dtype, minibatch_size)
    if key in _MODEL_CACHE and not force_reload:
        return _MODEL_CACHE[key]
    print(f'  Loading MapAnything from {MODEL_REPO_ID} ...')
    t0 = time.time()
    # Apache 2.0 weights variant. force_download=False uses cached snapshot.
    model = MapAnything.from_pretrained(
        MODEL_REPO_ID,
        cache_dir=str(drive_root),
        force_download=force_reload,
    )
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device).eval()
    _MODEL_CACHE[key] = model
    print(f'  model loaded in {time.time()-t0:.1f}s on {device}')
    return model

# ── Image loading ──────────────────────────────────────
def _collect_images(image_or_dir, max_n=64):
    """Collect a sorted list of image paths from a folder, video, or single image."""
    SUPP = {'.png', '.jpg', '.jpeg'}
    p = pathlib.Path(image_or_dir).resolve()
    if p.is_dir():
        files = sorted(
            x for x in p.rglob('*') if x.suffix.lower() in SUPP
        )
    elif p.is_file() and p.suffix.lower() in SUPP:
        files = [p]
    else:
        raise SystemExit(f'No images found at {p}')
    if not files:
        raise SystemExit(f'No images found at {p}')
    if len(files) > max_n:
        print(f'  WARN: {len(files)} images found; using first {max_n} (T4 VRAM limit)')
        files = files[:max_n]
    return files

def _load_video_frames(video_path, fps=1, max_frames=64):
    """Use ffmpeg to extract frames at the given fps from a video."""
    import subprocess
    out_dir = pathlib.Path(tempfile.mkdtemp(prefix='ma_video_')) if 'tempfile' in dir() \
              else pathlib.Path('/content/_ma_video')
    out_dir.mkdir(parents=True, exist_ok=True)
    pattern = out_dir / '%05d.png'
    cmd = [
        'ffmpeg', '-y', '-i', str(video_path),
        '-vf', f'fps={fps},scale=iw:ih',
        '-frames:v', str(max_frames),
        str(pattern),
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'ffmpeg frame extraction failed: {r.stderr[-500:]}')
    files = sorted(out_dir.glob('*.png'))
    print(f'  extracted {len(files)} frames @ {fps} fps from {pathlib.Path(video_path).name}')
    return files

import tempfile  # imported here so _load_video_frames can use it

def infer(image_paths, apply_mask=True, mask_edges=True,
         apply_confidence_mask=False, confidence_percentile=10,
         use_multiview_confidence=False, use_amp=True, amp_dtype='bf16',
         minibatch_size=1, memory_efficient_inference=True,
         ignore_calibration_inputs=False, ignore_depth_inputs=False,
         ignore_pose_inputs=False,
         ignore_depth_scale_inputs=False, ignore_pose_scale_inputs=False,
         stride=1):
    """Run MapAnything on a list of image paths. Returns the list of per-view
    prediction dicts (each with pts3d, pts3d_cam, intrinsics, camera_poses,
    depth_z, ray_directions, cam_trans, cam_quats, confidence, mask,
    non_ambiguous_mask, metric_scaling_factor, img_no_norm, etc.).
    See https://map-anything.github.io for the full output format.
    `stride=N` subsamples every Nth image (1 = use all). Critical for long
    videos where the view count would otherwise exceed VRAM."""
    model = get_model(amp_dtype=amp_dtype, minibatch_size=minibatch_size)
    # Apply stride before loading (avoids loading thousands of frames into RAM)
    if stride > 1:
        image_paths = list(image_paths)[::stride]
        print(f'  stride={stride}: using {len(image_paths)} of original images')
    print(f'  loading + preprocessing {len(image_paths)} images ...')
    # load_images handles DINOv2 normalization + resize to 518×518 internally
    views = load_images([str(p) for p in image_paths])
    print(f'  views loaded: {len(views)}')
    print(f'  MapAnything forward pass ...')
    t0 = time.time()
    with torch.inference_mode():
        predictions = model.infer(
            views,
            apply_mask=apply_mask,
            mask_edges=mask_edges,
            apply_confidence_mask=apply_confidence_mask,
            confidence_percentile=confidence_percentile,
            use_multiview_confidence=use_multiview_confidence,
            use_amp=use_amp,
            amp_dtype=amp_dtype,
            minibatch_size=minibatch_size,
            memory_efficient_inference=memory_efficient_inference,
            ignore_calibration_inputs=ignore_calibration_inputs,
            ignore_depth_inputs=ignore_depth_inputs,
            ignore_pose_inputs=ignore_pose_inputs,
            ignore_depth_scale_inputs=ignore_depth_scale_inputs,
            ignore_pose_scale_inputs=ignore_pose_scale_inputs,
        )
    print(f'  inference done in {time.time()-t0:.1f}s ({len(predictions)} views)')
    return predictions

def write_colmap(predictions, image_paths, output_dir,
                 voxel_fraction=0.01, voxel_size=None,
                 min_init_opacity=0.1):
    """Write MapAnything's per-view predictions to COLMAP format.
    COLMAP cameras.txt + images.txt + points3D.ply + a copy of the images.
    `min_init_opacity` filters out low-confidence points from the .ply
    (MapAnything's `non_ambiguous_mask` → init_opacity for gsplat)."""
    output_dir = pathlib.Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / 'images').mkdir(exist_ok=True)
    (output_dir / 'sparse' / '0').mkdir(parents=True, exist_ok=True)
    # Copy images
    import shutil
    for i, src in enumerate(image_paths):
        shutil.copy2(str(src), str(output_dir / 'images' / f'{i}.png'))
    n = len(predictions)
    # Cameras.txt (PINHOLE model, 1-indexed)
    cam_lines = []
    for i, pred in enumerate(predictions):
        K = pred['intrinsics'][0].cpu().numpy()  # (3, 3)
        fx, fy = K[0, 0], K[1, 1]
        cx, cy = K[0, 2], K[1, 2]
        cam_lines.append(f'{i+1} PINHOLE {fx:.6f} {fy:.6f} {cx:.6f} {cy:.6f}\n')
    (output_dir / 'sparse' / '0' / 'cameras.txt').write_text(''.join(cam_lines))
    # Images.txt (qw qx qy qz tx ty tz + image_name)
    img_lines = []
    for i, pred in enumerate(predictions):
        c2w = pred['camera_poses'][0].cpu().numpy()  # (4, 4) cam2world
        R = c2w[:3, :3]
        t = c2w[:3, 3]
        from scipy.spatial.transform import Rotation
        quat = Rotation.from_matrix(R).as_quat()  # [x, y, z, w]
        qx, qy, qz, qw = quat
        img_lines.append(
            f'{i+1} {qw:.6f} {qx:.6f} {qy:.6f} {qz:.6f} '
            f'{t[0]:.6f} {t[1]:.6f} {t[2]:.6f} {i+1} {i}.png\n\n'
        )
    (output_dir / 'sparse' / '0' / 'images.txt').write_text(''.join(img_lines))
    # Points3D.ply from the first view's dense point cloud,
    # filtered by non_ambiguous_mask (if available) for better 3DGS init
    pts = predictions[0]['pts3d'][0].cpu().numpy().reshape(-1, 3)
    if 'non_ambiguous_mask' in predictions[0]:
        mask = predictions[0]['non_ambiguous_mask'][0].cpu().numpy().reshape(-1)
        n_before = len(pts)
        # Treat mask as a soft opacity; keep only points with mask >= min_init_opacity
        pts = pts[mask >= min_init_opacity]
        print(f'  mask filter: {n_before:,} -> {len(pts):,} points (min_opacity={min_init_opacity})')
    n_pts = pts.shape[0]
    with open(output_dir / 'sparse' / '0' / 'points3D.ply', 'w') as f:
        f.write('ply\nformat ascii 1.0\n'
                f'element vertex {n_pts}\n'
                'property float x\nproperty float y\nproperty float z\n'
                'end_header\n')
        for p in pts:
            f.write(f'{p[0]:.4f} {p[1]:.4f} {p[2]:.4f}\n')
    print(f'  COLMAP dataset written: {n} views, {n_pts:,} points')
    return output_dir

def run_full_pipeline(image_paths, output_dir, train_3dgs=True,
                     gsplat_iterations=7000, voxel_fraction=0.01,
                     amp_dtype='bf16', minibatch_size=1,
                     do_drive_save=True, save_glb_export=True,
                     min_init_opacity=0.1, **infer_kwargs):
    """Full pipeline: MapAnything → COLMAP → gsplat → .ply (+ optional .glb)
    Returns (colmap_dir, ply_path, glb_path, n_views)"""
    image_paths = [pathlib.Path(p) for p in image_paths]
    output_dir = pathlib.Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    predictions = infer(image_paths, amp_dtype=amp_dtype,
                        minibatch_size=minibatch_size, **infer_kwargs)
    colmap_dir = write_colmap(predictions, image_paths,
                              output_dir / 'colmap',
                              voxel_fraction=voxel_fraction,
                              min_init_opacity=min_init_opacity)
    glb_path = None
    if save_glb_export:
        glb_path = save_glb(predictions, output_dir)
    ply = None
    if train_3dgs:
        ply = train_gsplat(colmap_dir, output_dir / 'gsplat',
                           iterations=gsplat_iterations)
    if do_drive_save and ply is not None:
        paths_to_save = [ply]
        if glb_path is not None:
            paths_to_save.append(glb_path)
        _mirror_to_drive(paths_to_save, str(output_dir / 'gsplat'))
    return colmap_dir, ply, glb_path, len(predictions)

def save_glb(predictions, output_dir, max_pts=200000):
    """Save a dense point cloud as .glb for direct viewing in any 3D viewer.
    Mirrors upstream `demo_inference_on_colmap_outputs.py --save_glb`.
    Subsamples to `max_pts` to keep file size reasonable (<50 MB)."""
    try:
        import trimesh
    except ImportError:
        print('  WARN: trimesh not installed, skipping GLB export')
        return None
    output_dir = pathlib.Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    # Concatenate all per-view points, color from each view's img
    all_pts = []
    all_cols = []
    for pred in predictions:
        pts = pred['pts3d'][0].cpu().numpy().reshape(-1, 3)
        # img_no_norm: (B, H, W, 3) in [0, 1]
        if 'img_no_norm' in pred:
            cols = pred['img_no_norm'][0].cpu().numpy().reshape(-1, 3)
        else:
            cols = np.ones_like(pts) * 0.5
        all_pts.append(pts)
        all_cols.append(cols)
    pts = np.concatenate(all_pts, axis=0)
    cols = np.concatenate(all_cols, axis=0)
    if len(pts) > max_pts:
        idx = np.random.choice(len(pts), max_pts, replace=False)
        pts = pts[idx]
        cols = cols[idx]
    pc = trimesh.PointCloud(vertices=pts, colors=(cols * 255).astype(np.uint8))
    glb_path = output_dir / 'reconstruction.glb'
    pc.export(str(glb_path))
    print(f'  GLB exported: {glb_path}  ({glb_path.stat().st_size//(1024*1024)} MB, {len(pts):,} pts)')
    return glb_path

def train_gsplat(colmap_dir, output_dir, iterations=7000):
    """Train a 3DGS scene from a COLMAP dataset using the gsplat simple_trainer."""
    import sys
    from gsplat.examples.simple_trainer import main as gsplat_main
    from argparse import Namespace
    output_dir = pathlib.Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f'  gsplat training ({iterations} iters) ...')
    t0 = time.time()
    # gsplat's main() reads CLI args. Use a Namespace to inject them.
    args = Namespace(
        data_dir=str(colmap_dir),
        result_dir=str(output_dir),
        data_factor=1,
        init_scale=0.1,  # for sparse point clouds from MapAnything
        steps=int(iterations),
        means_l2norm_reg=0.0,
        sh_degree=3,
        sh_degree_interval=1000,
        packed=False,
        antialiased=False,
        eval_steps=[] if not True else [-1],
        save_steps=[] if not True else [int(iterations)],
        disable_viewer=True,
        max_n_render_parallel=8,
    )
    try:
        gsplat_main(args)
    except SystemExit:
        pass  # gsplat calls sys.exit() at the end
    # Find the output PLY
    ply_candidates = sorted(output_dir.rglob('*.ply'))
    ply = max(ply_candidates, key=lambda p: p.stat().st_mtime) if ply_candidates else None
    if ply is None:
        raise FileNotFoundError(f'gsplat did not produce a .ply in {output_dir}')
    print(f'  gsplat training done in {time.time()-t0:.0f}s')
    print(f'  output .ply: {ply}  ({ply.stat().st_size//(1024*1024)} MB)')
    return ply

def _mirror_to_drive(paths, output_root, drive_subdir='MapAnything'):
    if not paths:
        return
    drive_base = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out') / drive_subdir
    try:
        drive_base.parent.mkdir(parents=True, exist_ok=True)
        for src in paths:
            try:
                _src = pathlib.Path(src) if not isinstance(src, pathlib.Path) else src
                dst = drive_base / _src.relative_to(pathlib.Path(output_root))
                dst.parent.mkdir(parents=True, exist_ok=True)
                if _src.resolve() == dst.resolve():
                    continue
                if dst.exists() and dst.stat().st_size == _src.stat().st_size:
                    continue
                shutil.copy2(str(_src), str(dst))
            except Exception as e:
                print(f'  WARN: drive mirror of {_src.name} failed: {e}')
        print(f'  drive mirror: {drive_base}')
    except Exception as e:
        print(f'  drive mirror skipped: {e}')

print('  pipeline ready: infer, write_colmap, train_gsplat, run_full_pipeline')


In [ ]:
#@title STEP 3 — Core helpers (single scene, batch, video frames)
"""
Re-exports the pipeline from STEP 2 and adds:
  • `run_single_scene(image_or_video_or_dir, output_dir, ...)` — convenience
  • `run_batch(input_dir, output_dir, ...)` — process multiple subdirs
  • `extract_video_frames(video_path, output_dir, fps)` — already imported
"""
import subprocess, json

def run_single_scene(image_or_video, output_dir, train_3dgs=True,
                     gsplat_iterations=7000, voxel_fraction=0.01,
                     min_init_opacity=0.1, save_glb_export=True,
                     video_fps=1, max_frames=64, amp_dtype='bf16',
                     minibatch_size=1, do_drive_save=True, **infer_kwargs):
    """Run the full MapAnything + gsplat pipeline on one image folder OR one video.
    Auto-detects which by file extension."""
    src = pathlib.Path(image_or_video).resolve()
    out = pathlib.Path(output_dir).resolve()
    out.mkdir(parents=True, exist_ok=True)
    if src.suffix.lower() in {'.mp4', '.mov', '.webm', '.avi', '.mkv', '.m4v'}:
        frames_dir = out / 'frames'
        frames_dir.mkdir(parents=True, exist_ok=True)
        if not any(frames_dir.glob('*.png')):
            image_paths = _load_video_frames(src, fps=video_fps, max_frames=max_frames)
        else:
            image_paths = sorted(frames_dir.glob('*.png'))
    else:
        if src.is_dir():
            image_paths = _collect_images(src)
        elif src.is_file():
            image_paths = [src]
        else:
            raise SystemExit(f'Input not found: {src}')
    colmap_dir, ply, glb_path, n = run_full_pipeline(
        image_paths, out,
        train_3dgs=train_3dgs,
        gsplat_iterations=gsplat_iterations,
        voxel_fraction=voxel_fraction,
        min_init_opacity=min_init_opacity,
        save_glb_export=save_glb_export,
        amp_dtype=amp_dtype,
        minibatch_size=minibatch_size,
        do_drive_save=do_drive_save,
        **infer_kwargs,
    )
    return ply

def run_batch(input_dir, output_dir, train_3dgs=True, gsplat_iterations=7000,
              min_init_opacity=0.1, save_glb_export=True,
              video_fps=1, max_frames=64, amp_dtype='bf16',
              minibatch_size=1, do_drive_save=True, **infer_kwargs):
    """Process every video or subdirectory of `input_dir` as a separate scene."""
    in_dir = pathlib.Path(input_dir).resolve()
    if not in_dir.exists():
        raise SystemExit(f'Input not found: {in_dir}')
    out_dir = pathlib.Path(output_dir).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    SUPP = {'.mp4', '.mov', '.webm', '.avi', '.mkv', '.m4v'}
    videos = sorted(p for p in in_dir.glob('*') if p.suffix.lower() in SUPP)
    subdirs = sorted(p for p in in_dir.iterdir() if p.is_dir() and any(p.iterdir()))
    scenes = [(p, p.stem) for p in videos] + [(p, p.name) for p in subdirs]
    if not scenes:
        raise SystemExit(f'No videos or subdirs found in {in_dir}')
    n_ok = 0
    for src, slug in scenes:
        scene_out = out_dir / slug
        scene_out.mkdir(parents=True, exist_ok=True)
        # Skip if .ply already exists
        existing_plys = list(scene_out.rglob('gsplat/*.ply'))
        if existing_plys and existing_plys[0].stat().st_size > 1024:
            print(f'  skip: {slug}  (already done)')
            continue
        try:
            run_single_scene(src, scene_out,
                             train_3dgs=train_3dgs,
                             gsplat_iterations=gsplat_iterations,
                             min_init_opacity=min_init_opacity,
                             save_glb_export=save_glb_export,
                             video_fps=video_fps, max_frames=max_frames,
                             amp_dtype=amp_dtype,
                             minibatch_size=minibatch_size,
                             do_drive_save=False, **infer_kwargs)
            n_ok += 1
        except Exception as e:
            print(f'  ERROR on {slug}: {type(e).__name__}: {e}')
    if do_drive_save and n_ok > 0:
        all_plys = sorted(out_dir.rglob('gsplat/*.ply'))
        all_glbs = sorted(out_dir.rglob('reconstruction.glb'))
        paths = all_plys + all_glbs
        if paths:
            _mirror_to_drive(paths, str(out_dir))
    print(f'\n  done: {n_ok} new scene(s) processed')
    return n_ok

print('  scene helpers ready: run_single_scene, run_batch')


In [ ]:
#@title STEP 4 — Launch Gradio UI
"""
Interactive Gradio UI:
  - Multi-file upload (any 1+ images) OR video upload
  - All MapAnything params exposed (apply_mask, mask_edges, confidence,
    use_multiview_confidence, AMP dtype, ignore_*_scale_inputs, stride, etc.)
  - Auto-runs gsplat after MapAnything for 3DGS output
  - Optional .glb export of the dense point cloud
  - Run button -> .ply + optional .glb outputs
  - Drive mirror toggle
  - Apache 2.0 license notice
"""
import os, sys, time, json, shutil, pathlib, traceback, tempfile
import gradio as gr

gr.TEMP_DIR = '/tmp/gradio_ma'
os.makedirs(gr.TEMP_DIR, exist_ok=True)

def _do_run(files, video_file, gsplat_iterations, voxel_fraction,
            apply_mask, mask_edges, use_multiview_confidence,
            apply_confidence_mask, confidence_percentile,
            ignore_calibration_inputs, ignore_depth_inputs,
            ignore_pose_inputs, ignore_depth_scale_inputs,
            ignore_pose_scale_inputs,
            amp_dtype, minibatch_size, stride,
            video_fps, max_frames, save_glb_export,
            min_init_opacity, do_drive_save,
            progress=gr.Progress()):
    try:
        tmp = pathlib.Path(tempfile.mkdtemp(prefix='ma_gradio_'))
        if video_file is not None:
            src = pathlib.Path(video_file if isinstance(video_file, str) else video_file.name)
            progress(0.1, 'Extracting video frames ...')
            image_paths = _load_video_frames(src, fps=int(video_fps), max_frames=int(max_frames))
        elif files:
            saved = []
            for f in files:
                src = pathlib.Path(f.name if hasattr(f, 'name') else f)
                dst = tmp / src.name
                shutil.copy2(str(src), str(dst))
                saved.append(str(dst))
            image_paths = _collect_images(tmp, max_n=int(max_frames))
        else:
            return 'Please upload a video or images.', None, None, None
        n_imgs = len(image_paths)
        progress(0.25, f'Running MapAnything on {n_imgs} images ...')
        colmap_dir, ply, glb_path, n_views = run_full_pipeline(
            image_paths, tmp,
            train_3dgs=True,
            gsplat_iterations=int(gsplat_iterations),
            voxel_fraction=float(voxel_fraction),
            amp_dtype=amp_dtype,
            minibatch_size=int(minibatch_size),
            do_drive_save=do_drive_save,
            save_glb_export=save_glb_export,
            min_init_opacity=float(min_init_opacity),
            apply_mask=apply_mask,
            mask_edges=mask_edges,
            use_multiview_confidence=use_multiview_confidence,
            apply_confidence_mask=apply_confidence_mask,
            confidence_percentile=float(confidence_percentile),
            ignore_calibration_inputs=ignore_calibration_inputs,
            ignore_depth_inputs=ignore_depth_inputs,
            ignore_pose_inputs=ignore_pose_inputs,
            ignore_depth_scale_inputs=ignore_depth_scale_inputs,
            ignore_pose_scale_inputs=ignore_pose_scale_inputs,
            stride=int(stride),
        )
        if ply is None:
            return 'Pipeline failed - see cell output for details.', None, None, None
        progress(1.0, 'Done')
        status = (f'Generated 3DGS scene from {n_views} images -> '
                  f'{ply.name}  ({ply.stat().st_size//(1024*1024)} MB)'
                  + (f'  +  reconstruction.glb' if glb_path else ''))
        return status, str(ply), _splat_viewer_html(str(ply), str(glb_path) if glb_path else None), (str(glb_path) if glb_path else None)
    except Exception as e:
        traceback.print_exc()
        return f'Error: {type(e).__name__}: {e}', None, None, None

def _splat_viewer_html(ply_path, glb_path=None):
    if not ply_path or not pathlib.Path(ply_path).exists():
        return '<p style="color:#888">No scene to preview</p>'
    p = pathlib.Path(ply_path)
    glb_html = ''
    if glb_path and pathlib.Path(glb_path).exists():
        gp = pathlib.Path(glb_path)
        glb_html = (f'<br><br>GLB: {gp.name}  ({gp.stat().st_size//(1024*1024)} MB) - '
                    f'<a href="https://playcanvas.com/viewer" target="_blank" '
                    f'style="color:#4af">playcanvas.com/viewer</a>')
    return (f'<div style="width:100%; height:240px; background:#1a1a1a; display:flex; '
            f'align-items:center; justify-content:center; color:#aaa; font-family:monospace; '
            f'padding:20px; text-align:center; border-radius:8px">'
            f'3DGS scene ready: {p.name}  ({p.stat().st_size//(1024*1024)} MB)'
            f'<br><small>Open the .ply in: '
            f'<a href="https://supersplat.dev" target="_blank" style="color:#4af">supersplat.dev</a> - '
            f'<a href="https://playcanvas.com/viewer" target="_blank" style="color:#4af">playcanvas.com/viewer</a> - '
            f'<a href="https://gsplat.tech" target="_blank" style="color:#4af">gsplat.tech</a>'
            f'{glb_html}'
            f'</small></div>')

with gr.Blocks(title='MapAnything - Universal 3DGS (Apache 2.0)') as demo:
    gr.Markdown(value=
        '## [MapAnything](https://map-anything.github.io/) - Universal 3DGS from Images\n'
        'Meta universal 3D reconstruction framework. One feed-forward transformer\n'
        'handles images-only, image+poses, image+intrinsics+depth, and more.\n'
        'Apache 2.0 - fully commercial-OK.\n'
        '* **Paper:** [arXiv 2509.13414](https://arxiv.org/abs/2509.13414) (3DV 2026) | '
          '**Code:** [facebookresearch/map-anything](https://github.com/facebookresearch/map-anything) (Apache 2.0) | '
          '**Weights:** [facebook/map-anything-apache](https://huggingface.co/facebook/map-anything-apache) (Apache 2.0)'
    )
    with gr.Row():
        with gr.Column(scale=1):
            files = gr.File(
                label='Images  ( 1+ .png / .jpg )',
                file_count='multiple',
                file_types=['.png', '.jpg', '.jpeg'],
            )
            video_file = gr.Video(
                label='...or upload a video  ( .mp4 / .mov / .webm )',
                sources=['upload'],
            )
            with gr.Accordion('Generation Settings', open=False):
                gsplat_iterations = gr.Slider(
                    1000, 30000, value=7000, step=1000,
                    label='gsplat iterations (3DGS training)',
                    info='Number of gsplat training iterations. 7000 is a good balance.'
                )
                voxel_fraction = gr.Slider(
                    0.001, 0.1, value=0.01, step=0.001,
                    label='voxel_fraction (init scale)',
                    info='Initial Gaussian scale as a fraction of the scene extent. 0.01 = 1% of IQR-based scene extent.'
                )
                min_init_opacity = gr.Slider(
                    0.0, 1.0, value=0.1, step=0.05,
                    label='min_init_opacity (point cloud filter)',
                    info='Drop MapAnything points with non_ambiguous_mask below this. Higher = cleaner PLY + faster gsplat.'
                )
                apply_mask = gr.Checkbox(
                    True,
                    label='apply_mask (filter non-ambiguous depth)',
                    info='Mask out regions where the depth prediction is ambiguous.'
                )
                mask_edges = gr.Checkbox(
                    True,
                    label='mask_edges (normal/depth edge mask)',
                    info='Mask out edge artifacts using normal and depth consistency.'
                )
                apply_confidence_mask = gr.Checkbox(
                    False,
                    label='apply_confidence_mask (filter low-conf pixels)',
                    info='Use the predicted confidence to mask out unreliable pixels.'
                )
                use_multiview_confidence = gr.Checkbox(
                    False,
                    label='use_multiview_confidence (cross-view depth consistency)',
                    info='Use multi-view depth consistency for confidence instead of learned. ~2x slower.'
                )
                confidence_percentile = gr.Slider(
                    0, 50, value=10, step=1,
                    label='confidence_percentile',
                    info='Mask out bottom N percentile of confidence. 10 = drop lowest 10%.'
                )
                with gr.Accordion('Advanced: Input Ignore Flags', open=False):
                    ignore_calibration_inputs = gr.Checkbox(False, label='ignore_calibration_inputs', info='Ignore any provided intrinsics/calibration; use only images + poses/depth.')
                    ignore_depth_inputs = gr.Checkbox(False, label='ignore_depth_inputs', info='Ignore any provided depth_z maps; predict from images only.')
                    ignore_pose_inputs = gr.Checkbox(False, label='ignore_pose_inputs', info='Ignore any provided camera_poses; predict from images only.')
                    ignore_depth_scale_inputs = gr.Checkbox(False, label='ignore_depth_scale_inputs', info='Ignore is_metric_scale on depth inputs; useful when depth is in arbitrary units.')
                    ignore_pose_scale_inputs = gr.Checkbox(False, label='ignore_pose_scale_inputs', info='Ignore is_metric_scale on pose inputs; useful for non-metric-scale priors.')
                amp_dtype = gr.Dropdown(
                    choices=['bf16', 'fp16', 'fp32'],
                    value='bf16',
                    label='AMP dtype',
                    info='bf16 = best speed/quality on Ampere+. fp16 = fallback for older GPUs. fp32 = slowest but most accurate.'
                )
                minibatch_size = gr.Slider(
                    1, 8, value=1, step=1,
                    label='minibatch_size',
                    info='Minibatch size for memory-efficient inference. 1 = lowest GPU memory.'
                )
                stride = gr.Slider(
                    1, 32, value=1, step=1,
                    label='stride (subsample every Nth image)',
                    info='Subsample every Nth image before inference. 1 = use all. 5 = use 1 of every 5 (for long videos).'
                )
                video_fps = gr.Slider(
                    0.5, 5, value=1, step=0.5,
                    label='Video frame rate (extracted)',
                    info='Frames per second to extract from the video. 1 fps is a good default.'
                )
                max_frames = gr.Slider(
                    4, 128, value=32, step=4,
                    label='Max frames',
                    info='Maximum number of frames to process. Lower = less VRAM.'
                )
                save_glb_export = gr.Checkbox(
                    True,
                    label='Also save .glb (dense point cloud)',
                    info='Export a colorized .glb of the dense point cloud. ~50 MB. Open in playcanvas.com/viewer or any glTF viewer.'
                )
                do_drive_save = gr.Checkbox(
                    True,
                    label='Mirror .ply to Google Drive',
                    info='Copy outputs to /content/drive/MyDrive/AEI_3D_Out/MapAnything/'
                )
            run_btn = gr.Button('Run', variant='primary')
        with gr.Column(scale=1):
            log = gr.Textbox(label='Status', lines=2, interactive=False)
            output = gr.File(label='Generated 3DGS .ply', interactive=False)
            glb_output = gr.File(label='Dense point cloud .glb (optional)', interactive=False)
            viewer = gr.HTML(label='3DGS Preview')
            gr.Markdown(
                '**Next steps for the .ply**\n'
                '* Open in [supersplat.dev](https://supersplat.dev) for cleanup + editing\n'
                '* Open the .glb in [playcanvas.com/viewer](https://playcanvas.com/viewer) for a quick 3D preview\n'
                '* Pipe into **SplatTransform_Colab** STEP 3 to compress to game-ready SOG/SPLAT/SPZ/GLB\n'
                '* Or load directly in Three.js / WebGPU'
            )
    run_btn.click(
        _do_run,
        inputs=[files, video_file, gsplat_iterations, voxel_fraction,
                apply_mask, mask_edges, use_multiview_confidence,
                apply_confidence_mask, confidence_percentile,
                ignore_calibration_inputs, ignore_depth_inputs,
                ignore_pose_inputs, ignore_depth_scale_inputs,
                ignore_pose_scale_inputs,
                amp_dtype, minibatch_size, stride,
                video_fps, max_frames, save_glb_export,
                min_init_opacity, do_drive_save],
        outputs=[log, output, viewer, glb_output],
    )
    def _welcome():
        return ('Ready. Upload 1+ images or a short video and click Run.')
    demo.load(_welcome, outputs=[log])

from IPython.display import clear_output
clear_output()
demo.queue(default_concurrency_limit=2).launch(
    share=False, prevent_thread_lock=True, inline=True,
    show_error=True, height=1100,
)
print('\n  Gradio UI running above (cell stays alive - do not re-run)')

In [ ]:
#@title STEP 5 — Keep alive + session summary
"""
Keeps the runtime alive for 12 h. Prints a session summary.
"""
import datetime, time, json, sys, pathlib, os, shutil
summary = {
    'timestamp'   : datetime.datetime.utcnow().isoformat() + 'Z',
    'python'      : sys.version.split()[0],
    'torch'       : None, 'cuda': None, 'gpus': [],
    'repo'        : str(REPO_DIR),
    'ffmpeg'      : shutil.which('ffmpeg') is not None,
    'model_cached': list(_MODEL_CACHE.keys()),
    'drive_cache' : str(pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/MapAnything')),
}
try:
    import torch
    summary['torch'] = torch.__version__
    summary['cuda']  = torch.version.cuda
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            summary['gpus'].append(f'{p.name}  {p.total_memory // (1024**3)} GB')
except Exception as e:
    summary['torch_error'] = str(e)
print(json.dumps(summary, indent=2))
print()
print('  • STEP 4 is the Gradio UI (interactive)')
print('  • STEP 6 is the Colab single-scene picker')
print('  • STEP 7 is the Colab batch (multiple scenes)')
print('  • Outputs land in OUTPUT_DIR and (if mirrored) in /content/drive/MyDrive/AEI_3D_Out/MapAnything/')
print()
print('  License: Apache 2.0 (code + weights) — fully commercial-OK.')
print()
print('  Cell will keep the runtime alive for 12 h unless you disconnect.')
print('  Press the stop button in the toolbar to release the GPU.')

_start = time.time()
while time.time() - _start < 43200:
    time.sleep(300)
    print(f'  [{int(time.time()-_start)//60} min] still running — close tab to stop')


In [ ]:
#@title STEP 6 — Quick test (Colab single-scene picker)
"""
Quick-test one video OR one image folder. Use this for verification
or for scripting (no Gradio UI). For interactive UI, use STEP 4.
"""
import time, pathlib

INPUT_PATH       = ''  #@param {type:'string', placeholder: '/content/my_video.mp4 or /content/my_photos/'}  # info='Either a video file or a folder of images.'
OUTPUT_DIR       = '/content/MapAnything_Out'  #@param {type:'string'}  # info='Where the COLMAP dataset + .ply + .glb will be written.'

GSPLAT_ITERATIONS = 7000  #@param {type:'slider', min:1000, max:30000, step:1000}  # info='Number of gsplat training iterations. 7000 is a good balance.'
VOXEL_FRACTION   = 0.01  #@param {type:'number'}  # info='Initial Gaussian scale as a fraction of scene extent. 0.01 = 1%.'
MIN_INIT_OPACITY = 0.1  #@param {type:'slider', min:0, max:1, step:0.05}  # info='Drop MapAnything points with non_ambiguous_mask below this.'
SAVE_GLB         = True  #@param {type:'boolean'}  # info='Also export a colorized .glb of the dense point cloud.'
STRIDE           = 1  #@param {type:'slider', min:1, max:32, step:1}  # info='Subsample every Nth image before inference. 1 = use all.'
APPLY_MASK       = True  #@param {type:'boolean'}  # info='Mask out regions where the depth prediction is ambiguous.'
MASK_EDGES       = True  #@param {type:'boolean'}  # info='Mask out edge artifacts using normal and depth consistency.'
USE_MV_CONFIDENCE= False  #@param {type:'boolean'}  # info='Use multi-view depth consistency for confidence. ~2x slower.'
APPLY_CONF_MASK  = False  #@param {type:'boolean'}  # info='Use predicted confidence to mask out unreliable pixels.'
CONF_PERCENTILE  = 10  #@param {type:'slider', min:0, max:50, step:1}  # info='Mask out bottom N percentile of confidence. 10 = drop lowest 10%.'
IGNORE_CALIB     = False  #@param {type:'boolean'}  # info='Ignore any provided intrinsics/calibration.'
IGNORE_DEPTH     = False  #@param {type:'boolean'}  # info='Ignore any provided depth_z maps.'
IGNORE_POSE      = False  #@param {type:'boolean'}  # info='Ignore any provided camera_poses.'
IGNORE_DEPTH_SCALE= False  #@param {type:'boolean'}  # info='Ignore is_metric_scale on depth inputs.'
IGNORE_POSE_SCALE= False  #@param {type:'boolean'}  # info='Ignore is_metric_scale on pose inputs.'
AMP_DTYPE        = 'bf16'  #@param {type:'string'}  # info='bf16 = best speed/quality on Ampere+. fp16 = older GPUs. fp32 = slowest.'
MINIBATCH_SIZE   = 1  #@param {type:'slider', min:1, max:8, step:1}  # info='Minibatch size for memory-efficient inference. 1 = lowest VRAM.'
VIDEO_FPS        = 1  #@param {type:'slider', min:0.5, max:5, step:0.5}  # info='Frames per second to extract from the video.'
MAX_FRAMES       = 32  #@param {type:'slider', min:4, max:128, step:4}  # info='Maximum number of frames to process.'
DO_DRIVE_SAVE    = True  #@param {type:'boolean'}  # info='Copy outputs to /content/drive/MyDrive/AEI_3D_Out/MapAnything/.'

if not INPUT_PATH.strip():
    raise SystemExit('Set INPUT_PATH to a video or a folder of images.')
src = pathlib.Path(INPUT_PATH).resolve()
if not src.exists():
    raise SystemExit(f'Input not found: {src}')
out = pathlib.Path(OUTPUT_DIR).resolve()
out.mkdir(parents=True, exist_ok=True)
print(f'  input    : {src}')
print(f'  output   : {out}')
print(f'  iters    : {GSPLAT_ITERATIONS}')
print()
t0 = time.time()
colmap_dir, ply, glb_path, n = run_full_pipeline(
    [src], out,
    train_3dgs=True,
    gsplat_iterations=GSPLAT_ITERATIONS,
    voxel_fraction=VOXEL_FRACTION,
    min_init_opacity=MIN_INIT_OPACITY,
    save_glb_export=SAVE_GLB,
    stride=STRIDE,
    apply_mask=APPLY_MASK,
    mask_edges=MASK_EDGES,
    use_multiview_confidence=USE_MV_CONFIDENCE,
    apply_confidence_mask=APPLY_CONF_MASK,
    confidence_percentile=CONF_PERCENTILE,
    ignore_calibration_inputs=IGNORE_CALIB,
    ignore_depth_inputs=IGNORE_DEPTH,
    ignore_pose_inputs=IGNORE_POSE,
    ignore_depth_scale_inputs=IGNORE_DEPTH_SCALE,
    ignore_pose_scale_inputs=IGNORE_POSE_SCALE,
    amp_dtype=AMP_DTYPE,
    minibatch_size=MINIBATCH_SIZE,
    video_fps=VIDEO_FPS, max_frames=MAX_FRAMES,
    do_drive_save=DO_DRIVE_SAVE,
)
elapsed = time.time() - t0
print(f'\n  done in {elapsed:.0f}s ({n} views)')
print(f'  output: {ply}  ({ply.stat().st_size//(1024*1024)} MB)')
if glb_path:
    print(f'  glb   : {glb_path}  ({glb_path.stat().st_size//(1024*1024)} MB)')


In [ ]:
#@title STEP 7 — Batch process multiple scenes
"""
Process every video or subdirectory in `INPUT_DIR` as a separate scene.
Each output scene has its own .ply + .glb. Already-done scenes are skipped.
"""
import time, pathlib

INPUT_DIR        = ''  #@param {type:'string', placeholder: '/content/my_scenes/'}  # info='Folder containing videos or scene subdirs.'
OUTPUT_DIR       = '/content/MapAnything_Out'  #@param {type:'string'}  # info="Where each scene's .ply + .glb will be written."

GSPLAT_ITERATIONS = 7000  #@param {type:'slider', min:1000, max:30000, step:1000}  # info='gsplat training iterations per scene.'
VOXEL_FRACTION   = 0.01  #@param {type:'number'}  # info='Initial Gaussian scale as a fraction of scene extent.'
MIN_INIT_OPACITY = 0.1  #@param {type:'slider', min:0, max:1, step:0.05}  # info='Drop MapAnything points with non_ambiguous_mask below this.'
SAVE_GLB         = True  #@param {type:'boolean'}  # info='Also export a colorized .glb per scene.'
STRIDE           = 1  #@param {type:'slider', min:1, max:32, step:1}  # info='Subsample every Nth image before inference. 1 = use all.'
APPLY_MASK       = True  #@param {type:'boolean'}  # info='Mask out regions where the depth prediction is ambiguous.'
MASK_EDGES       = True  #@param {type:'boolean'}  # info='Mask out edge artifacts using normal and depth consistency.'
USE_MV_CONFIDENCE= False  #@param {type:'boolean'}  # info='Use multi-view depth consistency for confidence.'
APPLY_CONF_MASK  = False  #@param {type:'boolean'}  # info='Use predicted confidence to mask out unreliable pixels.'
CONF_PERCENTILE  = 10  #@param {type:'slider', min:0, max:50, step:1}  # info='Mask out bottom N percentile of confidence.'
IGNORE_CALIB     = False  #@param {type:'boolean'}  # info='Ignore any provided intrinsics/calibration.'
IGNORE_DEPTH     = False  #@param {type:'boolean'}  # info='Ignore any provided depth_z maps.'
IGNORE_POSE      = False  #@param {type:'boolean'}  # info='Ignore any provided camera_poses.'
IGNORE_DEPTH_SCALE= False  #@param {type:'boolean'}  # info='Ignore is_metric_scale on depth inputs.'
IGNORE_POSE_SCALE= False  #@param {type:'boolean'}  # info='Ignore is_metric_scale on pose inputs.'
AMP_DTYPE        = 'bf16'  #@param {type:'string'}  # info='bf16 = best speed/quality on Ampere+. fp16 = older GPUs.'
MINIBATCH_SIZE   = 1  #@param {type:'slider', min:1, max:8, step:1}  # info='Minibatch size for memory-efficient inference.'
VIDEO_FPS        = 1  #@param {type:'slider', min:0.5, max:5, step:0.5}  # info='Frames per second to extract from videos.'
MAX_FRAMES       = 32  #@param {type:'slider', min:4, max:128, step:4}  # info='Maximum number of frames per scene.'
DO_DRIVE_SAVE    = True  #@param {type:'boolean'}  # info='Copy all scene .ply + .glb files to /content/drive/MyDrive/AEI_3D_Out/MapAnything/.'

if not INPUT_DIR.strip():
    raise SystemExit('Set INPUT_DIR to a folder of videos or subdirectories.')
in_dir = pathlib.Path(INPUT_DIR).resolve()
if not in_dir.exists():
    raise SystemExit(f'Input not found: {in_dir}')
out_dir = pathlib.Path(OUTPUT_DIR).resolve()
out_dir.mkdir(parents=True, exist_ok=True)
print(f'  input    : {in_dir}')
print(f'  output   : {out_dir}')
print()
t0 = time.time()
n = run_batch(
    in_dir, out_dir,
    gsplat_iterations=GSPLAT_ITERATIONS,
    voxel_fraction=VOXEL_FRACTION,
    min_init_opacity=MIN_INIT_OPACITY,
    save_glb_export=SAVE_GLB,
    stride=STRIDE,
    apply_mask=APPLY_MASK,
    mask_edges=MASK_EDGES,
    use_multiview_confidence=USE_MV_CONFIDENCE,
    apply_confidence_mask=APPLY_CONF_MASK,
    confidence_percentile=CONF_PERCENTILE,
    ignore_calibration_inputs=IGNORE_CALIB,
    ignore_depth_inputs=IGNORE_DEPTH,
    ignore_pose_inputs=IGNORE_POSE,
    ignore_depth_scale_inputs=IGNORE_DEPTH_SCALE,
    ignore_pose_scale_inputs=IGNORE_POSE_SCALE,
    amp_dtype=AMP_DTYPE,
    minibatch_size=MINIBATCH_SIZE,
    video_fps=VIDEO_FPS, max_frames=MAX_FRAMES,
    do_drive_save=DO_DRIVE_SAVE,
)
elapsed = time.time() - t0
print(f'\n  batch done: {n} scene(s) in {elapsed:.0f}s')
